In [1]:
import timm
import torch
import torch.nn as nn

import os
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader, random_split

from dataset import JetbotDataset

In [4]:
# ==========================================
# PATHS
# ==========================================

csv_path = "../data/dataset_clean.csv"
img_path = "../data/dataset_images_clean"

model_name = 'timm/mobilenetv4_conv_small_050.e3000_r224_in1k'

# ==========================================
# LOAD CSV
# ==========================================

df = pd.read_csv(csv_path, dtype={"frame_timestamp": str})

print(df.head())
print(f"\nDataset size: {len(df)}")

# ==========================================
# MODEL & TRANSFORMS
# ==========================================

model = timm.create_model(model_name, pretrained=True)

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=True)

# ==========================================
# FULL DATASET
# ==========================================

dataset = JetbotDataset(
    dataframe=df,
    image_dir=img_path,
    transform=transform
)

# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(17)
)

print(f"Train size: {len(train_dataset)}")
print(f"Test size : {len(test_dataset)}")

# ==========================================
# DATALOADERS
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# ==========================================
# SANITY CHECK
# ==========================================

images, targets = next(iter(train_loader))

print("Images shape :", images.shape)
print("Targets shape:", targets.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/dataset_clean.csv'

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    steering_loss_sum = 0.0
    mae_sum = 0.0
    dir_sum = 0.0
    dir_count = 0

    for images, targets in loader:

        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = (
            nn.functional.mse_loss(outputs[:, 0], targets[:, 0]) +
            2.0 * nn.functional.mse_loss(outputs[:, 1], targets[:, 1])
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # steering MAE
        mae_sum += torch.abs(outputs[:, 1] - targets[:, 1]).sum().item()

        # steering loss
        steering_loss_sum += nn.functional.l1_loss(
            outputs[:, 1], targets[:, 1]
        ).item()

        # directional accuracy
        mask = targets[:, 1].abs() > 0.05

        if mask.sum() > 0:
            dir_sum += (
                (torch.sign(outputs[mask, 1]) ==
                 torch.sign(targets[mask, 1]))
                .float()
                .sum()
                .item()
            )
            dir_count += mask.sum().item()

    n = len(loader.dataset)

    return (
        total_loss / len(loader),
        steering_loss_sum / len(loader),
        mae_sum / n,
        dir_sum / max(dir_count, 1)
    )

In [5]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    steering_loss_sum = 0.0
    mae_sum = 0.0
    dir_sum = 0.0
    dir_count = 0

    for images, targets in loader:

        images = images.to(device)
        targets = targets.to(device)

        outputs = model(images)

        loss = (
            nn.functional.mse_loss(outputs[:, 0], targets[:, 0]) +
            2.0 * nn.functional.mse_loss(outputs[:, 1], targets[:, 1])
        )

        total_loss += loss.item()

        mae_sum += torch.abs(outputs[:, 1] - targets[:, 1]).sum().item()

        steering_loss_sum += nn.functional.l1_loss(
            outputs[:, 1], targets[:, 1]
        ).item()

        mask = targets[:, 1].abs() > 0.05

        if mask.sum() > 0:
            dir_sum += (
                (torch.sign(outputs[mask, 1]) ==
                 torch.sign(targets[mask, 1]))
                .float()
                .sum()
                .item()
            )
            dir_count += mask.sum().item()

    n = len(loader.dataset)

    return (
        total_loss / len(loader),
        steering_loss_sum / len(loader),
        mae_sum / n,
        dir_sum / max(dir_count, 1)
    )

In [6]:
# model = timm.create_model(model_name, pretrained=True)

# for param in model.parameters():
#     param.requires_grad = False

# model.classifier = nn.Sequential(
#     nn.Linear(1280, 256),
#     nn.ReLU(),
#     nn.Dropout(0.2),
#     nn.Linear(256, 2),
#     nn.Tanh()
# )
# model = model.to(device)

# optimizer = torch.optim.Adam(
#     model.classifier.parameters(),
#     lr=1e-3
# )

In [7]:
model = timm.create_model(model_name, pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 2),
    nn.Tanh()
)
model = model.to(device)

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

In [8]:
num_epochs = 10

for epoch in tqdm(range(num_epochs)):

    train_loss, train_steer_loss, train_mae, train_dir = train_one_epoch(model, train_loader)
    val_loss, val_steer_loss, val_mae, val_dir = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train loss: {train_loss:.4f} | MAE: {train_mae:.4f} | STEER: {train_steer_loss:.4f} | DIR: {train_dir:.3f}")
    print(f"Val loss  : {val_loss:.4f} | MAE: {val_mae:.4f} | STEER: {train_steer_loss:.4f} | DIR: {val_dir:.3f}")
    print("-" * 40)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/10
Train loss: 0.5162 | MAE: 0.3339 | STEER: 0.3339 | DIR: 0.724
Val loss  : 0.4236 | MAE: 0.3175 | STEER: 0.3339 | DIR: 0.815
----------------------------------------
Epoch 2/10
Train loss: 0.3743 | MAE: 0.2603 | STEER: 0.2609 | DIR: 0.786
Val loss  : 0.3595 | MAE: 0.2713 | STEER: 0.2609 | DIR: 0.807
----------------------------------------
Epoch 3/10
Train loss: 0.3302 | MAE: 0.2477 | STEER: 0.2473 | DIR: 0.802
Val loss  : 0.3608 | MAE: 0.2518 | STEER: 0.2473 | DIR: 0.773
----------------------------------------
Epoch 4/10
Train loss: 0.3152 | MAE: 0.2276 | STEER: 0.2270 | DIR: 0.791
Val loss  : 0.3330 | MAE: 0.2343 | STEER: 0.2270 | DIR: 0.739
----------------------------------------
Epoch 5/10
Train loss: 0.3131 | MAE: 0.2305 | STEER: 0.2300 | DIR: 0.798
Val loss  : 0.3530 | MAE: 0.2442 | STEER: 0.2300 | DIR: 0.790
----------------------------------------
Epoch 6/10
Train loss: 0.2991 | MAE: 0.2195 | STEER: 0.2195 | DIR: 0.798
Val loss  : 0.3424 | MAE: 0.2503 | STEER: 0.219

In [9]:
torch.save({
    "model_state_dict": model.state_dict()
}, "../models/mobilenet_medium")

In [10]:
checkpoint = torch.load("../models/mobilenet_tiny", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [11]:
# model = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=True)

# # get model specific transforms (normalization, resize)
# data_config = timm.data.resolve_model_data_config(model)
# transforms = timm.data.create_transform(**data_config, is_training=False)

# output = model(transforms(img).unsqueeze(0))  # unsqueeze single image into batch of 1

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
model_name = 'timm/mobilenetv4_conv_small_050.e3000_r224_in1k'
model = timm.create_model(model_name, pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 2),
    nn.Tanh()
)
model = model.to(device)

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

In [9]:
# model.eval()

# # example input matching your training resolution
# example_input = torch.randn(1, 3, 224, 224).to(device)

# # create TorchScript model
# scripted_model = torch.jit.trace(model, example_input)

# # save
# scripted_model.save("../models/mobilenet_tiny_torchscript.pt")

# print("TorchScript model exported.")

TorchScript model exported.


In [7]:
model = timm.create_model(model_name, pretrained=True)

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

In [8]:
transform

Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)